# Module 10: Caching and Broadcasting

Run in Google Colab or a local PySpark environment.

In [ ]:
# Run this cell first
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("LearningModules").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Load datasets (adjust path if needed)
students_df = spark.read.csv("../data/students.csv", header=True, inferSchema=True)
enrollments_df = spark.read.csv("../data/enrollments.csv", header=True, inferSchema=True)

## Cache with Timing Comparison

Caching keeps a DataFrame in memory so repeated operations avoid recomputation.

In [ ]:
import time

# Join to create a larger DataFrame
joined_df = students_df.join(enrollments_df, on="student_id")

# Without cache
start = time.time()
joined_df.groupBy("course_id").agg(avg("gpa")).collect()
no_cache = time.time() - start
print(f"Without cache: {no_cache:.4f}s")

# Cache and materialize
joined_df.cache()
joined_df.count()  # materialize the cache

# With cache
start = time.time()
joined_df.groupBy("course_id").agg(avg("gpa")).collect()
with_cache = time.time() - start
print(f"With cache: {with_cache:.4f}s")
print(f"Speedup: {no_cache / with_cache:.1f}x")

## Persist with Storage Levels

`.persist()` gives fine-grained control over where data is stored.

In [ ]:
from pyspark import StorageLevel

# Unpersist previous cache
joined_df.unpersist()

# Available storage levels:
# MEMORY_ONLY - default for cache()
# MEMORY_AND_DISK - spills to disk if memory is full
# DISK_ONLY - stores only on disk
# MEMORY_ONLY_SER - serialized in memory (less space, more CPU)

# Persist with MEMORY_AND_DISK
joined_df.persist(StorageLevel.MEMORY_AND_DISK)
joined_df.count()  # materialize

print(f"Storage level: {joined_df.storageLevel}")
print(f"Is cached: {joined_df.is_cached}")

In [ ]:
# Compare storage levels
joined_df.unpersist()

# MEMORY_ONLY
joined_df.persist(StorageLevel.MEMORY_ONLY)
joined_df.count()
start = time.time()
joined_df.filter(col("gpa") > 3.0).count()
mem_only = time.time() - start
joined_df.unpersist()

# DISK_ONLY
joined_df.persist(StorageLevel.DISK_ONLY)
joined_df.count()
start = time.time()
joined_df.filter(col("gpa") > 3.0).count()
disk_only = time.time() - start
joined_df.unpersist()

print(f"MEMORY_ONLY: {mem_only:.4f}s")
print(f"DISK_ONLY:   {disk_only:.4f}s")

## Broadcast Join

When one DataFrame is small, broadcasting it to all worker nodes avoids expensive shuffle operations.

In [ ]:
# Regular join
start = time.time()
regular_join = enrollments_df.join(students_df, on="student_id")
regular_join.count()
regular_time = time.time() - start

# Broadcast join (broadcast the smaller DataFrame)
start = time.time()
broadcast_join = enrollments_df.join(broadcast(students_df), on="student_id")
broadcast_join.count()
broadcast_time = time.time() - start

print(f"Regular join:   {regular_time:.4f}s")
print(f"Broadcast join: {broadcast_time:.4f}s")

In [ ]:
# Verify broadcast is used in the plan
broadcast_join.explain()

## Accumulators

Accumulators are shared variables that workers can add to. Useful for counting events during processing.

In [ ]:
# Create an accumulator to count high-GPA students
high_gpa_count = sc.accumulator(0)

def count_high_gpa(row):
    global high_gpa_count
    if row["gpa"] and row["gpa"] > 3.5:
        high_gpa_count.add(1)
    return row

# Process each row (use foreach for side effects)
students_df.foreach(count_high_gpa)

print(f"Students with GPA > 3.5: {high_gpa_count.value}")

## Practice Problem 1: Cache Effectiveness

Cache the students DataFrame, run 5 consecutive `.count()` operations, and time the total with and without caching.

<details><summary>Hint</summary>Unpersist first, time 5 counts, then cache, materialize, and time 5 more counts.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
students_df.unpersist()

start = time.time()
for _ in range(5):
    students_df.count()
no_cache_time = time.time() - start

students_df.cache()
students_df.count()  # materialize

start = time.time()
for _ in range(5):
    students_df.count()
cache_time = time.time() - start

print(f"Without cache (5 counts): {no_cache_time:.4f}s")
print(f"With cache (5 counts):    {cache_time:.4f}s")

## Practice Problem 2: Broadcast a Lookup Table

Create a small lookup DataFrame with course names and broadcast-join it with enrollments.

<details><summary>Hint</summary>Create a small DataFrame with createDataFrame(), then use broadcast() in the join.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
# Create a small lookup table
courses_lookup = spark.createDataFrame([
    ("CS101", "Intro to CS"),
    ("CS201", "Data Structures"),
    ("CS301", "Algorithms")
], ["course_id", "course_name"])

# Broadcast join
enriched = enrollments_df.join(broadcast(courses_lookup), on="course_id", how="left")
enriched.show()
enriched.explain()

## Practice Problem 3: Accumulator for Validation

Use an accumulator to count how many enrollment records have a null course_id.

<details><summary>Hint</summary>Create an accumulator, use foreach to check each row's course_id for None.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
null_course_count = sc.accumulator(0)

def check_null_course(row):
    global null_course_count
    if row["course_id"] is None:
        null_course_count.add(1)

enrollments_df.foreach(check_null_course)
print(f"Enrollments with null course_id: {null_course_count.value}")

## Practice Problem 4: Persist and Unpersist

Persist the enrollments DataFrame with MEMORY_AND_DISK, verify it is cached, run a query, then unpersist and verify.

<details><summary>Hint</summary>Use .persist(StorageLevel.MEMORY_AND_DISK), check .is_cached, run a query, then .unpersist().</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
enrollments_df.persist(StorageLevel.MEMORY_AND_DISK)
enrollments_df.count()  # materialize
print(f"Is cached: {enrollments_df.is_cached}")
print(f"Storage level: {enrollments_df.storageLevel}")

# Run a query using the cached data
enrollments_df.groupBy("course_id").count().show()

# Unpersist
enrollments_df.unpersist()
print(f"Is cached after unpersist: {enrollments_df.is_cached}")